### Native RAG Fusion Pipeline (Without `rag_fusion.py`)

This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) Fusion pipeline built **entirely with native LangChain components and LCEL (LangChain Expression Language)**, without needing any custom wrapper class or `rag_fusion.py`.

**Why no separate class is needed:**
- **Sub-Query Generation**: Handled directly by `ChatPromptTemplate | llm.with_structured_output(...)`.
- **Parallel Multi-Query Retrieval**: Handled natively by LangChain's **`retriever.map()`**, which invokes any retriever concurrently across a list of queries.
- **Reciprocal Rank Fusion (RRF)**: A lightweight standalone function that re-ranks documents using the reciprocal rank formula $RRF = \sum \frac{1}{\text{rank} + 60}$.
- **Complete Pipeline**: Composed seamlessly as an end-to-end LCEL chain.

**Steps covered:**
1. Load source PDF (`notebooklm_rag.pdf`)
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create base retriever
5. Implement native RAG Fusion using `retriever.map()` + RRF
6. Retrieve and inspect fused documents
7. Augment context and generate the final answer with LLM
8. Bonus: End-to-End One-Line LCEL Chain

### Imports & Environment Setup

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from pydantic import BaseModel, Field

# Load API keys from .env file
load_dotenv()

C:\Users\sujat\AppData\Local\Temp\ipykernel_7668\228696441.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### Step 1 - Load the Source PDF

`PyPDFLoader` reads `notebooklm_rag.pdf` and returns document pages.

In [2]:
loader = PyPDFLoader("notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF.")

Loaded 3 page(s) from the PDF.


### Step 2 - Split Documents into Chunks

We split the pages into overlapping chunks using `RecursiveCharacterTextSplitter`.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunk(s).")

Split into 19 chunk(s).


### Step 3 - Embeddings & ChromaDB Vector Store

We use HuggingFace embeddings (`sentence-transformers/all-MiniLM-L6-v2`) and store the chunks in ChromaDB.

In [4]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="notebooklm_rag_native"
)

print("Vector store created successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2034.57it/s]


Vector store created successfully.


### Step 4 - Base Retriever Setup

We set up a standard similarity search retriever returning the top 3 documents per query.

In [5]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

### Step 5 - Native RAG Fusion (No `rag_fusion.py` Needed!)

Here, we implement RAG Fusion natively using simple building blocks:
1. **Pydantic Schema**: Defines the structured output schema for the LLM to generate sub-queries.
2. **Sub-Query Generation Chain**: Uses LCEL to generate sub-queries from the input question.
3. **Reciprocal Rank Fusion (RRF)**: A clean function that merges and re-ranks documents based on reciprocal rank scoring.
4. **`retriever.map()`**: LangChain's built-in mapping runnable that invokes the retriever on each generated query in parallel.

In [6]:
# 1. Schema for structured sub-query generation
class SubQuerySchema(BaseModel):
    sub_queries: list[str] = Field(
        ..., 
        description="List of sub-queries generated from the main query to enhance retrieval."
    )

# 2. Initialize LLM (ChatGroq using GROQ_API_KEY from .env, or fallback to ChatOpenAI)
if os.getenv("GROQ_API_KEY"):
    llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0)
else:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. Sub-query generation prompt & chain
subquery_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that generates sub-queries from a main query to enhance retrieval."),
    ("user", "Given the main query: '{main_query}', generate {num_subqueries} sub-queries that can be used to retrieve relevant documents.")
])

# LCEL chain returning a list of query strings: ["subquery 1", "subquery 2", ...]
subquery_chain = (
    subquery_prompt 
    | llm.with_structured_output(SubQuerySchema) 
    | (lambda res: res.sub_queries)
)

In [7]:
# 4. Standalone Reciprocal Rank Fusion (RRF) function
def reciprocal_rank_fusion(retrieved_docs: list[list[Document]], k: int = 60, top_k: int = 3) -> list[Document]:
    """
    Applies Reciprocal Rank Fusion (RRF) across multiple retrieved document lists.
    Formula: RRF_score = sum(1 / (rank + k))
    """
    doc_scores: dict[str, tuple[float, Document]] = {}

    for doc_list in retrieved_docs:
        for rank, doc in enumerate(doc_list, start=1):
            key = doc.page_content
            rrf_score = 1.0 / (rank + k)
            
            if key in doc_scores:
                prev_score, prev_doc = doc_scores[key]
                doc_scores[key] = (prev_score + rrf_score, prev_doc)
            else:
                doc_scores[key] = (rrf_score, doc)

    # Sort descending by accumulated RRF score
    sorted_docs = sorted(doc_scores.values(), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in sorted_docs[:top_k]]

In [8]:
# 5. Composite Native RAG Fusion Retriever Chain using LCEL:
# subquery_chain -> retriever.map() -> reciprocal_rank_fusion
rag_fusion_retriever = (
    subquery_chain 
    | retriever.map() 
    | (lambda docs_list: reciprocal_rank_fusion(docs_list, k=60, top_k=3))
)

### Step 6 - Test Retrieval & Inspect Results

Let's test the native RAG Fusion chain with a user query and observe the generated sub-queries and fused documents.

In [9]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# Preview the sub-queries first
generated_subqueries = subquery_chain.invoke({"main_query": query, "num_subqueries": 2})
print("Generated Sub-Queries:")
for i, sq in enumerate(generated_subqueries, 1):
    print(f"  {i}. {sq}")

# Run the complete RAG Fusion retrieval
fused_docs = rag_fusion_retriever.invoke({"main_query": query, "num_subqueries": 2})

print(f"\nRetrieved {len(fused_docs)} fused document(s):")
for i, doc in enumerate(fused_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)

Generated Sub-Queries:
  1. What methods does NotebookLM use to index and search uploaded documents?
  2. How does NotebookLM extract and rank relevant content from user-provided files?

Retrieved 3 fused document(s):

--- Document 1 ---
vectors exist in the same semantic space, making cosine similarity a reliable measure of relevance.
The system then performs a nearest-neighbor search against the vector index to retrieve the top-k most
relevant document chunks. In practice, NotebookLM likely uses a combination of dense retrieval (vector
similarity) and sparse retrieval such as keyword-based BM25 matching, through a technique known as hybrid

--- Document 2 ---
model can reference the specific chunks it used to generate an answer.
3. How NotebookLM Processes Documents

--- Document 3 ---
rather than simple keyword matching. The vectors are stored in a vector index that supports efficient
nearest-neighbor search, enabling fast retrieval even across very large document collections.
4. Qu

### Step 7 - Augmentation & Generation

We format the fused documents into a unified context block and generate the grounded final answer using the LLM.

In [10]:
# Build context from fused documents
context = "\n\n".join([doc.page_content for doc in fused_docs])

generation_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Generation chain
generation_chain = generation_prompt | llm | StrOutputParser()

# Generate answer
response = generation_chain.invoke({"context": context, "question": query})
print(response)

NotebookLM first breaks each uploaded document into chunks and encodes each chunk into a dense vector (an embedding).  
All of these vectors are placed in a vector index that supports fast nearest‑neighbor lookup.  

When a user asks a question, the query is encoded with the same model to produce a query vector.  
NotebookLM then compares this query vector to the stored chunk vectors using cosine similarity (and, in practice, often combines this dense‑vector match with a sparse keyword‑based BM25 match in a hybrid retrieval step).  

The system retrieves the top‑k most similar document chunks from the index and uses those chunks as the source of information for generating the answer.


### Step 8 - Bonus: Complete End-to-End One-Line LCEL Chain

With LCEL, the entire process (Query -> Sub-Queries -> Multi-Retrieval -> RRF -> Context -> Answer) can be composed into a single unified runnable chain:

In [11]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

# Unified End-to-End RAG Fusion Chain
full_rag_fusion_chain = (
    {
        "context": (
            (lambda x: {"main_query": x["question"], "num_subqueries": x.get("num_subqueries", 2)})
            | rag_fusion_retriever
            | format_docs
        ),
        "question": lambda x: x["question"]
    }
    | generation_prompt
    | llm
    | StrOutputParser()
)

# Test the unified chain
answer = full_rag_fusion_chain.invoke({
    "question": "How does NotebookLM retrieve relevant information from uploaded documents?",
    "num_subqueries": 2
})

print(answer)

NotebookLM first breaks each uploaded document into chunks and encodes each chunk into a dense vector (an embedding).  
All of these vectors are stored in a vector index that supports fast nearest‑neighbor lookup.  

When a user asks a question, the query is encoded with the same model to produce a query vector.  
NotebookLM then compares this query vector to the stored chunk vectors using cosine similarity (and may also combine this dense similarity with a sparse BM25 keyword match in a hybrid fashion).  

The system performs a nearest‑neighbor search to pull the top‑k most similar document chunks, which are then used to generate the answer.
